# 2 pamoka – Microsoft Agent Framework tyrinėjimas

**Microsoft Agent Framework (MAF)** yra vieningas AI agentų kūrimo karkasas. Jis siūlo aiškią, kompozicinę architektūrą su keturiais pagrindiniais komponentais:

- **Klientas** – jungiasi prie AI modelio taško ir tvarko komunikaciją
- **Agentas** – apgaubia klientą su instrukcijomis ir įrankių apibrėžimais
- **Įrankiai** – praplečia agento galimybes su modelio iškviečiamomis pasirinktinių funkcijų funkcijomis
- **Seansas** – palaiko pokalbio istoriją daugkartinėms sąveikoms

Šioje pamokoje sukursime **kelionių rezervavimo agentą**, kuris tikrins kelionės tikslų prieinamumą naudodamas šias sąvokas.


## Sąranka


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Agentų sistemos architektūros supratimas

Microsoft Agent Framework naudoja sluoksniuotą architektūrą:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Klientas** – `AzureAIProjectAgentProvider` jungiasi prie Azure OpenAI diegimo. Jis tvarko autentifikavimą, užklausų formatavimą ir atsakymų analizę.
2. **Agentas** – Sukuriamas iš kliento per `provider.create_agent()`, agentas sujungia modelio prieigą su instrukcijomis (sisteminiu pranešimu) ir įrankiais.
3. **Įrankiai** – Python funkcijos su `@tool` dekoratoriumi, kuriuos agentas gali kvieti atlikti veiksmus ar gauti duomenis.
4. **Sesija** – `AgentSession` objektas (sukurtas per `agent.create_session()`), kuris saugo pokalbio istoriją, leidžiantį daugiakartines dialogo sesijas, kai agentas atsimena ankstesnį kontekstą.

Sukurkime kiekvieną sluoksnį po žingsnio.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Įrankių pridėjimas naudojant @tool dekoratorių

Įrankiai leidžia agentams imtis veiksmų, kurie nėra tik teksto generavimas. `@tool` dekoratorius paverčia įprastą Python funkciją į kažką, ką agentas gali kviesti.

Svarbiausi dalykai:
- Naudokite `Annotated[type, "description"]`, kad modelis suprastų kiekvieną parametrą.
- Docstring tampa įrankio aprašymu, kurį mato modelis.
- `approval_mode="never_require"` reiškia, kad įrankis veikia automatiškai be naudotojo patvirtinimo.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Agentas su Įrankiais Kūrimas

Dabar sujungiame klientą, instrukcijas ir įrankius į agentą. `instructions` veikia kaip sistemos užklausa — jie apibrėžia agento asmenybę ir elgesį.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Daugkartiniai pokalbiai su seansais

`AgentSession` (sukuriamas per `agent.create_session()`) seka visus pranešimus pokalbyje. Perduodami tą patį seansą kiekvienam `agent.run()` kvietimui, agentas turi prieigą prie visos pokalbio istorijos ir gali kreiptis į ankstesnius pranešimus.

Mes perduodame `tools=[check_destination_availability]`, kad agentas galėtų kaskart pokalbio metu iškviesti mūsų prieinamumo tikrintuvą.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Santrauka

Šiame pamokoje jūs susipažinote su keturiais Microsoft Agent Framework pamatiniais elementais:

| Sąvoka | Ką sužinojote |
|---------|------------------|
| **Klientas** | `AzureAIProjectAgentProvider` jungiasi prie Azure OpenAI su kredencialine autentifikacija |
| **Agentas** | `provider.create_agent()` apjungia modelio jungtį su instrukcijomis ir pavadinimu |
| **Įrankiai** | `@tool` dekoratorius leidžia agentui kviesti Python funkcijas |
| **Seansas** | `agent.create_session()` palaiko pokalbio istoriją per kelis turus |

Šie statybiniai blokai sudaro agentus, kurie gali vesti natūralias diskusijas, kviesti išorines funkcijas ir palaikyti kontekstą — tai pagrindas sudėtingesniems agentiniams modeliams vėlesnėse pamokose.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:  
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, atkreipkite dėmesį, kad automatizuotuose vertimuose gali būti klaidų ar netikslumų. Pirminis dokumentas gimtąja kalba laikomas autoritetingu šaltiniu. Esant svarbiai informacijai, rekomenduojama pasinaudoti profesionalaus žmogaus vertimo paslaugomis. Mes neatsakome už bet kokius nesusipratimus ar neteisingą interpretavimą, kilusius dėl šio vertimo naudojimo.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
